In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.home() / "Climate-Change-Awareness-Dashboard"

PRECIPITATION_DIR = PROJECT_DIR / "precipitation"

INPUT_PATH = (
    PRECIPITATION_DIR
    / "data"
    / "processed"
    / "01_consolidated_before_final_feature_alignment.csv"
)

OUTPUT_DIR = PRECIPITATION_DIR / "data" / "processed"

df = pd.read_csv(INPUT_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (253939, 33)


In [3]:
# Calculate grid-cell-specific monthly rainfall normal

baseline = df[df["year"].between(1961, 1990)]

rainfall_normal = (
    baseline.groupby(["lat", "lon", "month"])["monthly_accumulated_rain"]
    .mean()
    .reset_index(name="rainfall_normal")
)

df = df.merge(
    rainfall_normal,
    on=["lat", "lon", "month"],
    how="left",
    validate="many_to_one"
)

In [4]:
# Calculate grid-cell-specific monthly rainfall standard deviation

rainfall_std = (
    baseline.groupby(["lat", "lon", "month"])["monthly_accumulated_rain"]
    .std()
    .reset_index(name="rainfall_std_normal")
)

df = df.merge(
    rainfall_std,
    on=["lat", "lon", "month"],
    how="left",
    validate="many_to_one"
)

In [5]:
# Calculate monthly rainfall anomaly

df["rainfall_anomaly"] = (
    df["monthly_accumulated_rain"]
    - df["rainfall_normal"]
)

In [6]:
# Generate rainfall anomaly lag features

df = df.sort_values(["lat", "lon", "year", "month"])

for lag in [1, 2, 3]:
    df[f"rainfall_anomaly_lag_{lag}"] = (
        df.groupby(["lat", "lon"])["rainfall_anomaly"].shift(lag)
    )

In [7]:
# Generate 3-month rolling rainfall anomaly

df = df.sort_values(["lat", "lon", "year", "month"])

df["rainfall_anomaly_rolling_3m"] = (
    df.groupby(["lat", "lon"])["rainfall_anomaly"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=3).mean())
)

In [8]:
# Generate 6-month rolling rainfall anomaly

df["rainfall_anomaly_rolling_6m"] = (
    df.groupby(["lat", "lon"])["rainfall_anomaly"]
    .transform(lambda x: x.shift(1).rolling(window=6, min_periods=6).mean())
)

In [9]:
# Generate 3-month rolling standard deviation of rainfall anomaly

df["rainfall_anomaly_std_3m"] = (
    df.groupby(["lat", "lon"])["rainfall_anomaly"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=3).std())
)

In [10]:
# Correct rainfall lag and rolling features using calendar-month alignment

df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"], day=1)
)

feature_frames = []

for (lat, lon), group in df.groupby(["lat", "lon"]):

    group = group.sort_values("date").set_index("date")

    # Create a complete monthly timeline
    full_dates = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="MS"
    )

    aligned = group.reindex(full_dates)

    anomaly = aligned["rainfall_anomaly"]

    features = pd.DataFrame(index=full_dates)

    features["lat"] = lat
    features["lon"] = lon

    # Previous calendar-month anomalies
    features["rainfall_anomaly_lag_1"] = anomaly.shift(1)
    features["rainfall_anomaly_lag_2"] = anomaly.shift(2)
    features["rainfall_anomaly_lag_3"] = anomaly.shift(3)

    # Previous 3- and 6-calendar-month averages
    previous = anomaly.shift(1)

    features["rainfall_anomaly_rolling_3m"] = (
        previous.rolling(3, min_periods=3).mean()
    )

    features["rainfall_anomaly_rolling_6m"] = (
        previous.rolling(6, min_periods=6).mean()
    )

    # Previous 3-calendar-month standard deviation
    features["rainfall_anomaly_std_3m"] = (
        previous.rolling(3, min_periods=3).std()
    )

    features.index.name = "date"
    feature_frames.append(features.reset_index())

# Combine features for all locations
corrected_features = pd.concat(feature_frames, ignore_index=True)

# Remove earlier, incorrectly aligned feature columns
rainfall_lag_cols = [
    "rainfall_anomaly_lag_1",
    "rainfall_anomaly_lag_2",
    "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m",
    "rainfall_anomaly_rolling_6m",
    "rainfall_anomaly_std_3m"
]

df = df.drop(columns=rainfall_lag_cols)

# Merge corrected features
df = df.merge(
    corrected_features,
    on=["lat", "lon", "date"],
    how="left",
    validate="one_to_one"
)

# Final verification of all rainfall-related columns
rainfall_cols = [
    "rainfall_normal",
    "rainfall_std_normal",
    "rainfall_anomaly"
] + rainfall_lag_cols

print("Dataset shape:", df.shape)

print("\nMissing values in rainfall columns:")
print(df[rainfall_cols].isna().sum())

print("\nRainfall columns:")
print(rainfall_cols)

Dataset shape: (253939, 29)

Missing values in rainfall columns:
rainfall_normal                   0
rainfall_std_normal               0
rainfall_anomaly                  0
rainfall_anomaly_lag_1          329
rainfall_anomaly_lag_2          628
rainfall_anomaly_lag_3          924
rainfall_anomaly_rolling_3m     969
rainfall_anomaly_rolling_6m    1908
rainfall_anomaly_std_3m         969
dtype: int64

Rainfall columns:
['rainfall_normal', 'rainfall_std_normal', 'rainfall_anomaly', 'rainfall_anomaly_lag_1', 'rainfall_anomaly_lag_2', 'rainfall_anomaly_lag_3', 'rainfall_anomaly_rolling_3m', 'rainfall_anomaly_rolling_6m', 'rainfall_anomaly_std_3m']


In [11]:
# Find the first record where all rainfall lag/rolling features are available

rainfall_input_cols = [
    "rainfall_anomaly_lag_1",
    "rainfall_anomaly_lag_2",
    "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m",
    "rainfall_anomaly_rolling_6m",
    "rainfall_anomaly_std_3m"
]

# Identify rows where every rainfall input feature is available
complete_rainfall_rows = df[
    df[rainfall_input_cols].notna().all(axis=1)
]

# Find the earliest complete record for each location
first_complete = (
    complete_rainfall_rows
    .sort_values(["lat", "lon", "year", "month"])
    .groupby(["lat", "lon"], as_index=False)
    .first()
)

print("First complete record for each location:")
print(first_complete[["lat", "lon", "year", "month"]])

print("\nOverall earliest complete record:")
print(
    first_complete
    .sort_values(["year", "month"])
    [["lat", "lon", "year", "month"]]
    .head(1)
)

print("\nNumber of locations with complete rainfall inputs:",
      len(first_complete))
print("Total locations:", df[["lat", "lon"]].drop_duplicates().shape[0])

First complete record for each location:
      lat   lon  year  month
0     8.5  77.5  1951      7
1     9.5  76.5  1951      7
2     9.5  77.5  1951      7
3     9.5  78.5  1951      7
4    10.5  76.5  1951      7
..    ...   ...   ...    ...
278  34.5  74.5  1955     11
279  34.5  75.5  1955      7
280  34.5  76.5  1951      7
281  34.5  77.5  1951      7
282  34.5  78.5  1951      7

[283 rows x 4 columns]

Overall earliest complete record:
   lat   lon  year  month
0  8.5  77.5  1951      7

Number of locations with complete rainfall inputs: 283
Total locations: 283


In [12]:
# Generate temperature anomaly lag features

df = df.sort_values(["lat", "lon", "year", "month"])

for lag in [1, 2, 3]:
    df[f"temperature_anomaly_lag_{lag}"] = (
        df.groupby(["lat", "lon"])["tmean_anom"].shift(lag)
    )

In [13]:
# Generate 3-month rolling temperature anomaly

df["temperature_anomaly_rolling_3m"] = (
    df.groupby(["lat", "lon"])["tmean_anom"]
    .transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=3).mean()
    )
)

In [14]:
# Check for missing calendar months within each location's timeline

df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"], day=1)
)

gap_summary = []

for (lat, lon), group in df.groupby(["lat", "lon"]):

    dates = pd.DatetimeIndex(group["date"].sort_values().unique())

    expected_dates = pd.date_range(
        start=dates.min(),
        end=dates.max(),
        freq="MS"
    )

    missing_dates = expected_dates.difference(dates)

    gap_summary.append({
        "lat": lat,
        "lon": lon,
        "first_record": dates.min(),
        "last_record": dates.max(),
        "missing_months_in_middle": len(missing_dates),
        "missing_months": list(missing_dates.strftime("%Y-%m"))
    })

gap_df = pd.DataFrame(gap_summary)

print("Locations with gaps in the middle:",
      (gap_df["missing_months_in_middle"] > 0).sum())

print("Total missing calendar months:",
      gap_df["missing_months_in_middle"].sum())

print("\nLocations with missing months:")
print(
    gap_df[gap_df["missing_months_in_middle"] > 0].to_string(index=False)
)

Locations with gaps in the middle: 8
Total missing calendar months: 375

Locations with missing months:
 lat  lon first_record last_record  missing_months_in_middle                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [15]:
# Save a backup of the current dataset

BACKUP_PATH = OUTPUT_DIR / "01_consolidated_before_final_feature_alignment.csv"

df.to_csv(BACKUP_PATH, index=False)

print("Backup saved:", BACKUP_PATH)

Backup saved: /home/sharanya/Climate-Change-Awareness-Dashboard/precipitation/data/processed/01_consolidated_before_final_feature_alignment.csv


In [6]:
# Generate cyclical month features

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [7]:
# Check whether all required model input columns exist

model_input_cols = [
    "year",
    "lat",
    "lon",
    "month_sin",
    "month_cos",
    "rainfall_anomaly_lag_1",
    "rainfall_anomaly_lag_2",
    "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m",
    "rainfall_anomaly_rolling_6m",
    "rainfall_anomaly_std_3m",
    "temperature_anomaly_lag_1",
    "temperature_anomaly_lag_2",
    "temperature_anomaly_lag_3",
    "temperature_anomaly_rolling_3m"
]

target_col = "rainfall_anomaly"

missing_inputs = [
    col for col in model_input_cols
    if col not in df.columns
]

print("Missing model input columns:", missing_inputs)
print("Target column available:", target_col in df.columns)
print("Number of model inputs available:",
      len(model_input_cols) - len(missing_inputs), "/ 15")

Missing model input columns: []
Target column available: True
Number of model inputs available: 15 / 15


In [8]:
# Save a backup before calendar alignment

BACKUP_PATH = OUTPUT_DIR / "02_before_calendar_alignment.csv"

df.to_csv(BACKUP_PATH, index=False)

print("Backup saved:", BACKUP_PATH)

Backup saved: /home/sharanya/Climate-Change-Awareness-Dashboard/precipitation/data/processed/02_before_calendar_alignment.csv


In [9]:
# STEP 4: Calendar-align all rainfall and temperature features

df = df.copy()

df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"], day=1)
)

feature_cols = [
    "rainfall_anomaly_lag_1",
    "rainfall_anomaly_lag_2",
    "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m",
    "rainfall_anomaly_rolling_6m",
    "rainfall_anomaly_std_3m",
    "temperature_anomaly_lag_1",
    "temperature_anomaly_lag_2",
    "temperature_anomaly_lag_3",
    "temperature_anomaly_rolling_3m"
]

# Process each grid location independently
for (lat, lon), group in df.groupby(["lat", "lon"]):

    group = group.sort_values("date")

    # Create a complete monthly timeline
    aligned = (
        group.set_index("date")
        .reindex(
            pd.date_range(
                group["date"].min(),
                group["date"].max(),
                freq="MS"
            )
        )
    )

    # Rainfall anomaly features
    rain = aligned["rainfall_anomaly"]

    for lag in [1, 2, 3]:
        aligned[f"rainfall_anomaly_lag_{lag}"] = rain.shift(lag)

    previous_rain = rain.shift(1)

    aligned["rainfall_anomaly_rolling_3m"] = (
        previous_rain.rolling(3, min_periods=3).mean()
    )

    aligned["rainfall_anomaly_rolling_6m"] = (
        previous_rain.rolling(6, min_periods=6).mean()
    )

    aligned["rainfall_anomaly_std_3m"] = (
        previous_rain.rolling(3, min_periods=3).std()
    )

    # Temperature anomaly features
    temp = aligned["tmean_anom"]

    for lag in [1, 2, 3]:
        aligned[f"temperature_anomaly_lag_{lag}"] = temp.shift(lag)

    aligned["temperature_anomaly_rolling_3m"] = (
        temp.shift(1).rolling(3, min_periods=3).mean()
    )

    # Copy corrected values back only to existing dataset records
    for col in feature_cols:
        df.loc[group.index, col] = (
            aligned.loc[group["date"], col].to_numpy()
        )

# Save the corrected dataset persistently
WORKING_PATH = OUTPUT_DIR / "02_feature_engineering_working.csv"

df.to_csv(WORKING_PATH, index=False)

print("Calendar alignment completed.")
print("Corrected dataset saved to:", WORKING_PATH)
print("Dataset shape:", df.shape)

Calendar alignment completed.
Corrected dataset saved to: /home/sharanya/Climate-Change-Awareness-Dashboard/precipitation/data/processed/02_feature_engineering_working.csv
Dataset shape: (253939, 35)


In [12]:
# Display first 5 rows of the updated dataset

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

df.head()
print(df.columns.tolist())

['year', 'month', 'lat', 'lon', 'location', 'nearest_city', 'state', 'monthly_accumulated_rain', 'monthly_mean_rain', 'valid_days', 'tmax_mean', 'tmin_mean', 'tmean_mean', 'tmax_normal', 'tmin_normal', 'tmean_normal', 'tmax_anom', 'tmin_anom', 'tmean_anom', 'rainfall_normal', 'rainfall_std_normal', 'rainfall_anomaly', 'date', 'rainfall_anomaly_lag_1', 'rainfall_anomaly_lag_2', 'rainfall_anomaly_lag_3', 'rainfall_anomaly_rolling_3m', 'rainfall_anomaly_rolling_6m', 'rainfall_anomaly_std_3m', 'temperature_anomaly_lag_1', 'temperature_anomaly_lag_2', 'temperature_anomaly_lag_3', 'temperature_anomaly_rolling_3m', 'month_sin', 'month_cos']


In [13]:
# Remove temporary date column
df.drop(columns=["date"], inplace=True)

# Save updated working dataset
df.to_csv(WORKING_PATH, index=False)

print("Date column removed.")
print("Dataset shape:", df.shape)

Date column removed.
Dataset shape: (253939, 34)


In [14]:
# Final missing-value analysis

model_input_cols = [
    "year",
    "lat",
    "lon",
    "month_sin",
    "month_cos",
    "rainfall_anomaly_lag_1",
    "rainfall_anomaly_lag_2",
    "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m",
    "rainfall_anomaly_rolling_6m",
    "rainfall_anomaly_std_3m",
    "temperature_anomaly_lag_1",
    "temperature_anomaly_lag_2",
    "temperature_anomaly_lag_3",
    "temperature_anomaly_rolling_3m"
]

target_col = "rainfall_anomaly"

print("Missing values in model inputs:")
print(df[model_input_cols].isna().sum())

print("\nMissing values in target:")
print(df[target_col].isna().sum())

# Identify records with all required inputs and target available
complete_mask = (
    df[model_input_cols + [target_col]]
    .notna()
    .all(axis=1)
)

print("\nTotal records:", len(df))
print("Complete records:", complete_mask.sum())
print("Incomplete records:", (~complete_mask).sum())

print("\nLocations represented:", df[["lat", "lon"]].drop_duplicates().shape[0])
print("Locations with complete records:",
      df.loc[complete_mask, ["lat", "lon"]].drop_duplicates().shape[0])

Missing values in model inputs:
year                                 0
lat                                  0
lon                                  0
month_sin                            0
month_cos                            0
rainfall_anomaly_lag_1             329
rainfall_anomaly_lag_2             628
rainfall_anomaly_lag_3             924
rainfall_anomaly_rolling_3m        969
rainfall_anomaly_rolling_6m       1908
rainfall_anomaly_std_3m            969
temperature_anomaly_lag_1          329
temperature_anomaly_lag_2          628
temperature_anomaly_lag_3          924
temperature_anomaly_rolling_3m     969
dtype: int64

Missing values in target:
0

Total records: 253939
Complete records: 252031
Incomplete records: 1908

Locations represented: 283
Locations with complete records: 283


In [15]:
# Analyse incomplete modelling records

model_input_cols = [
    "year", "lat", "lon", "month_sin", "month_cos",
    "rainfall_anomaly_lag_1",
    "rainfall_anomaly_lag_2",
    "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m",
    "rainfall_anomaly_rolling_6m",
    "rainfall_anomaly_std_3m",
    "temperature_anomaly_lag_1",
    "temperature_anomaly_lag_2",
    "temperature_anomaly_lag_3",
    "temperature_anomaly_rolling_3m"
]

incomplete = df[
    df[model_input_cols].isna().any(axis=1)
].copy()

# Missing values by location
location_missing = (
    incomplete.groupby(["lat", "lon"])
    .size()
    .reset_index(name="incomplete_records")
    .sort_values("incomplete_records", ascending=False)
)

print("Incomplete records:", len(incomplete))
print("\nLocations affected:", len(location_missing))
print("\nIncomplete records by location:")
print(location_missing.to_string(index=False))

print("\nIncomplete records by year:")
print(incomplete.groupby("year").size().to_string())

Incomplete records: 1908

Locations affected: 283

Incomplete records by location:
 lat  lon  incomplete_records
27.5 96.5                  55
34.5 75.5                  48
34.5 74.5                  40
33.5 73.5                  37
34.5 78.5                  30
28.5 96.5                  24
34.5 76.5                  12
34.5 77.5                  12
25.5 94.5                   6
25.5 92.5                   6
25.5 91.5                   6
25.5 90.5                   6
25.5 88.5                   6
25.5 87.5                   6
25.5 86.5                   6
25.5 85.5                   6
25.5 84.5                   6
25.5 83.5                   6
25.5 82.5                   6
25.5 81.5                   6
25.5 80.5                   6
25.5 78.5                   6
25.5 93.5                   6
26.5 86.5                   6
26.5 85.5                   6
26.5 84.5                   6
26.5 83.5                   6
26.5 82.5                   6
26.5 81.5                   6
26.5 80.5        

In [16]:
# Analyse incomplete modelling records

model_input_cols = [
    "year", "lat", "lon", "month_sin", "month_cos",
    "rainfall_anomaly_lag_1",
    "rainfall_anomaly_lag_2",
    "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m",
    "rainfall_anomaly_rolling_6m",
    "rainfall_anomaly_std_3m",
    "temperature_anomaly_lag_1",
    "temperature_anomaly_lag_2",
    "temperature_anomaly_lag_3",
    "temperature_anomaly_rolling_3m"
]

incomplete = df[
    df[model_input_cols].isna().any(axis=1)
].copy()

# Missing values by location
location_missing = (
    incomplete.groupby(["lat", "lon"])
    .size()
    .reset_index(name="incomplete_records")
    .sort_values("incomplete_records", ascending=False)
)

print("\nIncomplete records by year:")
print(incomplete.groupby("year").size().to_string())


Incomplete records by year:
year
1951    1674
1952       6
1953       5
1954       5
1955      15
1956       2
1957      19
1959      11
1961      38
1962       9
1963      11
1964       5
1965       8
1966      25
1967      12
1968      11
1969      10
1992       6
1995       1
1996       5
2000       6
2003      12
2009       4
2010       8


In [17]:
# Summarise incomplete records by location and year

incomplete = df[
    df[model_input_cols].isna().any(axis=1)
].copy()

location_year_summary = (
    incomplete.groupby(["lat", "lon", "year"])
    .size()
    .reset_index(name="incomplete_records")
)

print("Incomplete records by location and year:")
print(location_year_summary.to_string(index=False))

Incomplete records by location and year:
 lat  lon  year  incomplete_records
 8.5 77.5  1951                   6
 9.5 76.5  1951                   6
 9.5 77.5  1951                   6
 9.5 78.5  1951                   6
10.5 76.5  1951                   6
10.5 77.5  1951                   6
10.5 78.5  1951                   6
10.5 79.5  1951                   6
11.5 76.5  1951                   6
11.5 77.5  1951                   6
11.5 78.5  1951                   6
11.5 79.5  1951                   6
12.5 75.5  1951                   6
12.5 76.5  1951                   6
12.5 77.5  1951                   6
12.5 78.5  1951                   6
12.5 79.5  1951                   6
13.5 75.5  1951                   6
13.5 76.5  1951                   6
13.5 77.5  1951                   6
13.5 78.5  1951                   6
13.5 79.5  1951                   6
14.5 74.5  1951                   6
14.5 75.5  1951                   6
14.5 76.5  1951                   6
14.5 77.5  1951        

In [20]:
import pandas as pd

# Create a monthly date for every record
df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"], day=1)
)

gap_results = []

for (lat, lon), group in df.groupby(["lat", "lon"]):

    actual_months = pd.PeriodIndex(group["date"], freq="M").unique().sort_values()

    # Ignore the initial Jan-Jun 1951 warm-up period
    analysis_start = max(
        actual_months.min(),
        pd.Period("1951-07", freq="M")
    )
    analysis_end = actual_months.max()

    expected_months = pd.period_range(
        start=analysis_start,
        end=analysis_end,
        freq="M"
    )

    missing_months = expected_months.difference(actual_months)

    if len(missing_months) == 0:
        continue

    # Group consecutive missing months into ranges
    ranges = []
    start = previous = missing_months[0]

    for month in missing_months[1:]:
        if month.ordinal == previous.ordinal + 1:
            previous = month
        else:
            ranges.append(f"{start} to {previous}")
            start = previous = month

    ranges.append(f"{start} to {previous}")

    gap_results.append({
        "latitude": lat,
        "longitude": lon,
        "missing_months": len(missing_months),
        "expected_months": len(expected_months),
        "missing_percentage": round(
            len(missing_months) / len(expected_months) * 100, 2
        ),
        "missing_periods": "; ".join(ranges)
    })

gap_df = pd.DataFrame(gap_results)

gap_df = gap_df.sort_values(
    "missing_months",
    ascending=False
).reset_index(drop=True)

from IPython.display import display, HTML

display(HTML(
    gap_df.to_html(index=False, escape=False)
))

display(HTML("""
<style>
.output table {
    width: 100%;
    table-layout: fixed;
}
.output th, .output td {
    white-space: normal !important;
    overflow-wrap: anywhere;
    text-align: left;
    padding: 8px;
}
.output th:nth-child(6), .output td:nth-child(6) {
    min-width: 350px;
}
</style>
"""))

latitude,longitude,missing_months,expected_months,missing_percentage,missing_periods
28.5,96.5,122,684,17.84,1991-12 to 1991-12; 1993-01 to 2002-12; 2009-10 to 2009-10
34.5,74.5,101,848,11.91,1957-01 to 1959-04; 1959-09 to 1959-09; 1959-11 to 1961-08; 1961-10 to 1962-06; 1962-10 to 1963-05; 1963-11 to 1964-09; 1964-11 to 1965-06; 1965-09 to 1966-05; 1966-11 to 1967-01; 1968-01 to 1968-02
27.5,96.5,73,830,8.80,1957-02 to 1957-02; 1961-02 to 1961-02; 1966-11 to 1966-11; 1967-04 to 1967-09; 1968-09 to 1968-10; 1995-11 to 1995-11; 1997-01 to 1999-12; 2001-01 to 2002-12; 2009-10 to 2009-10
34.5,75.5,67,852,7.86,1957-01 to 1959-02; 1959-12 to 1961-02; 1961-08 to 1961-08; 1961-11 to 1962-01; 1962-12 to 1963-03; 1963-12 to 1964-03; 1964-06 to 1964-07; 1964-09 to 1964-09; 1964-11 to 1965-03; 1965-06 to 1965-06; 1965-11 to 1966-01; 1966-03 to 1966-04
33.5,73.5,6,894,0.67,1953-07 to 1953-07; 1954-07 to 1954-07; 1954-09 to 1954-09; 1957-04 to 1957-04; 1961-02 to 1961-02; 1966-01 to 1966-01
34.5,78.5,4,894,0.45,1952-01 to 1952-01; 1957-04 to 1957-04; 1961-02 to 1961-02; 1966-01 to 1966-01
34.5,76.5,1,894,0.11,1961-02 to 1961-02
34.5,77.5,1,894,0.11,1961-02 to 1961-02


In [21]:
# Model input features
FEATURES = [
    "year",
    "lat",
    "lon",
    "month_sin",
    "month_cos",
    "rainfall_anomaly_lag_1",
    "rainfall_anomaly_lag_2",
    "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m",
    "rainfall_anomaly_rolling_6m",
    "rainfall_anomaly_std_3m",
    "temperature_anomaly_lag_1",
    "temperature_anomaly_lag_2",
    "temperature_anomaly_lag_3",
    "temperature_anomaly_rolling_3m"
]

TARGET = "rainfall_anomaly"

# Create a separate dataset containing only model columns
model_df = df[FEATURES + [TARGET]].copy()

print("Model dataset shape:", model_df.shape)
print("Number of input features:", len(FEATURES))
print("Target:", TARGET)

display(model_df.head())

Model dataset shape: (253939, 16)
Number of input features: 15
Target: rainfall_anomaly


,year,lat,lon,month_sin,month_cos,rainfall_anomaly_lag_1,rainfall_anomaly_lag_2,rainfall_anomaly_lag_3,rainfall_anomaly_rolling_3m,rainfall_anomaly_rolling_6m,rainfall_anomaly_std_3m,temperature_anomaly_lag_1,temperature_anomaly_lag_2,temperature_anomaly_lag_3,temperature_anomaly_rolling_3m,rainfall_anomaly
0,1951,8.5,77.5,0.500000,8.660254e-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.792233
1,1951,8.5,77.5,0.866025,5.000000e-01,17.792233,NaN,NaN,NaN,NaN,NaN,0.227,NaN,NaN,NaN,-20.624075
2,1951,8.5,77.5,1.000000,6.123234e-17,-20.624075,17.792233,NaN,NaN,NaN,NaN,-0.562,0.227,NaN,NaN,-10.911211
3,1951,8.5,77.5,0.866025,-5.000000e-01,-10.911211,-20.624075,17.792233,-4.581018,NaN,19.975152,0.034,-0.562,0.227,-0.100333,89.001956
4,1951,8.5,77.5,0.500000,-8.660254e-01,89.001956,-10.911211,-20.624075,19.155556,NaN,60.683396,-0.982,0.034,-0.562,-0.503333,22.288481


In [22]:
MODEL_PATH = OUTPUT_DIR / "03_model_training_dataset.csv"

model_df.to_csv(MODEL_PATH, index=False)

print("Initial model dataset saved:", MODEL_PATH)

Initial model dataset saved: /home/sharanya/Climate-Change-Awareness-Dashboard/precipitation/data/processed/03_model_training_dataset.csv


In [23]:
# Check complete records for all model features and target
required_columns = FEATURES + [TARGET]

complete_mask = model_df[required_columns].notna().all(axis=1)

location_counts = (
    model_df.assign(is_complete=complete_mask)
    .groupby(["lat", "lon"])
    .agg(
        total_records=("is_complete", "size"),
        complete_records=("is_complete", "sum")
    )
)

location_counts["incomplete_records"] = (
    location_counts["total_records"] -
    location_counts["complete_records"]
)

display(location_counts.describe().round(2))

print("Total locations:", len(location_counts))
print("Locations with zero complete records:",
      (location_counts["complete_records"] == 0).sum())

,total_records,complete_records,incomplete_records
count,283.00,283.00,283.00
mean,897.31,890.57,6.74
std,24.50,28.07,5.02
min,562.00,538.00,6.00
25%,900.00,894.00,6.00
50%,900.00,894.00,6.00
75%,900.00,894.00,6.00
max,900.00,894.00,55.00


Total locations: 283
Locations with zero complete records: 0


In [24]:
# Remove rows with missing values in any model feature or target
model_df.dropna(
    subset=FEATURES + [TARGET],
    inplace=True
)

print("Final model dataset shape:", model_df.shape)
print("Remaining missing values:",
      model_df[FEATURES + [TARGET]].isna().sum().sum())

# Save the cleaned model dataset
MODEL_PATH = OUTPUT_DIR / "03_model_training_dataset.csv"
model_df.to_csv(MODEL_PATH, index=False)

print("Saved:", MODEL_PATH)

Final model dataset shape: (252031, 16)
Remaining missing values: 0
Saved: /home/sharanya/Climate-Change-Awareness-Dashboard/precipitation/data/processed/03_model_training_dataset.csv
